# ML Assignment 2 - Classification Models on Spambase (Google Colab)

**Course:** Machine Learning, M.Tech (AIML), BITS Pilani WILP

This notebook follows the assignment steps:

1. Step 1 - choose one public classification dataset (>= 12 features, >= 500 rows).
2. Step 2 - train 5 classifiers on it and compute Accuracy, AUC, Precision, Recall, F1 and MCC.
3. Step 3 - save the models (`model/*.joblib`) and the test data (`test_data.csv`).
4. Step 5 - produce the comparison table for the README.

The Streamlit app (`app.py`) is a separate file. It loads the saved models and evaluates them on an uploaded test CSV.

**Run all cells in order (Runtime > Run all).** The last cell downloads the model zip, `test_data.csv` and `model_comparison.csv`.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix, classification_report, roc_curve)

RANDOM_STATE = 42
os.makedirs("model", exist_ok=True)
print("scikit-learn version:", sklearn.__version__)

## 1. Dataset (Step 1)

**Spambase** from the UCI Machine Learning Repository (https://archive.ics.uci.edu/dataset/94/spambase).

- 4,601 emails (>= 500 required)
- 57 numeric features (>= 12 required): word frequencies, character frequencies and capital-letter run statistics
- Binary target: 1 = spam, 0 = not spam

In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"

word_cols = [f"word_freq_{w}" for w in (
    "make address all 3d our over remove internet order mail receive will people report "
    "addresses free business email you credit your font 000 money hp hpl george 650 lab labs "
    "telnet 857 data 415 85 technology 1999 parts pm direct cs meeting original project re "
    "edu table conference").split()]
char_cols = [f"char_freq_{c}" for c in ("semicolon", "paren", "bracket", "exclamation", "dollar", "hash")]
cap_cols = ["capital_run_length_average", "capital_run_length_longest", "capital_run_length_total"]
columns = word_cols + char_cols + cap_cols + ["spam"]

df = pd.read_csv(URL, header=None, names=columns)
print(df.shape)
assert df.shape[1] - 1 >= 12 and df.shape[0] >= 500
df.head()

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
counts = df["spam"].value_counts().rename({0: "not spam", 1: "spam"})
print(counts)
counts.plot(kind="bar", title="Class distribution")
plt.show()

## 2. Train / test split

A stratified 80/20 split keeps the class ratio equal in both parts. The 20% test portion is saved as `test_data.csv` and is what the Streamlit app evaluates. Models are fit on the training portion only.

In [ ]:
FEATURES = [c for c in df.columns if c != "spam"]
X, y = df[FEATURES], df["spam"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("train:", X_train.shape, " test:", X_test.shape)
print("train spam ratio: %.3f   test spam ratio: %.3f" % (y_train.mean(), y_test.mean()))

## 3. Models (Step 2)

Each model is a scikit-learn `Pipeline`, so the preprocessing is saved inside the model file and the app needs no separate scaler.

| Model | Preprocessing | Notes |
|---|---|---|
| Logistic Regression | StandardScaler | `max_iter=2000` so the solver converges |
| Decision Tree | none (scale-invariant) | `max_depth=8` limits overfitting |
| KNN | StandardScaler (distance-based) | `n_neighbors=5` |
| Naive Bayes | none | `GaussianNB` |
| SVM | StandardScaler | RBF kernel, `probability=True` for AUC |

In [ ]:
models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    "Decision Tree": Pipeline([("clf", DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE))]),
    "KNN": Pipeline([("scaler", StandardScaler()),
                     ("clf", KNeighborsClassifier(n_neighbors=5))]),
    "Naive Bayes": Pipeline([("clf", GaussianNB())]),
    "Support Vector Machine": Pipeline([("scaler", StandardScaler()),
                                        ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))]),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print("trained:", name)

## 4. Evaluation metrics (Accuracy, AUC, Precision, Recall, F1, MCC)

In [ ]:
def evaluate(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    return {
        "Accuracy":  accuracy_score(y, pred),
        "AUC":       roc_auc_score(y, proba),
        "Precision": precision_score(y, pred),
        "Recall":    recall_score(y, pred),
        "F1":        f1_score(y, pred),
        "MCC":       matthews_corrcoef(y, pred),
    }

results = pd.DataFrame({n: evaluate(m, X_test, y_test) for n, m in models.items()}).T
results.index.name = "ML Model Name"
results = results.round(4)
results

In [ ]:
# Train-set metrics, to see which models overfit
train_results = pd.DataFrame({n: evaluate(m, X_train, y_train) for n, m in models.items()}).T.round(4)
print("Training-set metrics")
display(train_results)
print("Train - test accuracy gap")
display((train_results["Accuracy"] - results["Accuracy"]).round(4).to_frame("gap"))

### Markdown table for the README

Copy the output of the next cell into the *Models used* section of `README.md`.

In [ ]:
def to_markdown(df):
    header = "| ML Model Name | " + " | ".join(df.columns) + " |"
    sep = "|---|" + "---|" * len(df.columns)
    rows = ["| " + idx + " | " + " | ".join(f"{v:.4f}" for v in r) + " |" for idx, r in df.iterrows()]
    return "\n".join([header, sep] + rows)

print(to_markdown(results))
print("\nBest model by MCC:", results["MCC"].idxmax())

## 5. Confusion matrices and ROC curves

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, m) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, m.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Not spam", "Spam"], yticklabels=["Not spam", "Spam"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 5))
for name, m in models.items():
    fpr, tpr, _ = roc_curve(y_test, m.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=f"{name} (AUC={results.loc[name, 'AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves on the test set"); plt.legend(loc="lower right"); plt.show()

In [ ]:
for name, m in models.items():
    print("=" * 60); print(name)
    print(classification_report(y_test, m.predict(X_test), target_names=["Not spam", "Spam"], digits=4))

## 6. Save models and test data (Step 3)

`app.py` expects these exact file names.

In [ ]:
FILES = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib",
}
for name, fname in FILES.items():
    joblib.dump(models[name], os.path.join("model", fname))

test_data = X_test.copy()
test_data["spam"] = y_test.values
test_data.to_csv("test_data.csv", index=False)
results.to_csv("model_comparison.csv")

# Reload check: saved models must reproduce the metrics without retraining
reloaded_test = pd.read_csv("test_data.csv")
for name, fname in FILES.items():
    reloaded = joblib.load(os.path.join("model", fname))
    got = evaluate(reloaded, reloaded_test[FEATURES], reloaded_test["spam"])
    assert all(abs(got[k] - results.loc[name, k]) < 1e-3 for k in got), name
print("Saved:", sorted(os.listdir("model")), "+ test_data.csv", test_data.shape)

## 7. Download everything

Downloads a zip of `model/`, plus `test_data.csv` and `model_comparison.csv`. Put these in your GitHub repo next to `app.py`, `requirements.txt` and `README.md`.

In [ ]:
import shutil
shutil.make_archive("model_files", "zip", ".", "model")
try:
    from google.colab import files
    files.download("model_files.zip")
    files.download("test_data.csv")
    files.download("model_comparison.csv")
except ImportError:
    print("Not running on Colab - files are in the current directory.")

## 8. Observations for the README

Fill in the observations after looking at your own results above. Points to check:

- Which model has the highest MCC and AUC? Is the gap to the others large or small?
- Precision vs recall: a spam filter usually values precision (do not block real mail).
- Train vs test gap: a large gap means overfitting.
- Naive Bayes assumes independent features, and word frequencies are correlated.
- SVM and KNN need scaled features, and the pipelines handle this.